# PRATYAKSHA · Step 5b — Laya (System-1), Claude Opus 5.5 (R7), and the final test month

This replaces cells 4–7 of the Step 5 notebook. As in the mid-sem plan:
- **Laya** (421M, non-generative System-1 decision model) is fine-tuned on months 1–4 to choose the match and
  decide "alone or a person", then routed through the same R6 rules.
- **Claude Opus 5.5** is the frontier reference, on **every** dev and test line, with the same evidence.
- Then **every system runs once on the held-out test month**.

Before you start: Colab **Secrets** (key icon, left sidebar) must contain `ANTHROPIC_API_KEY` with *Notebook access* on.
Upload `pratyaksha_step5b.zip` to *Colab Notebooks*. Runtime **T4 GPU**. About 2.5–3 h in total; every cell can be
re-run safely after a reconnect (finished runs are skipped). Always run cell 1 first in a new session.

## 1. Setup
**Why:** Drive, the new code, libraries (now including `laya`), data. **Expect:** `35 passed`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
%cd "/content/drive/MyDrive/Colab Notebooks"
!unzip -o -q pratyaksha_step5b.zip
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!pip install -q -U transformers accelerate sentence-transformers rank-bm25 pytest peft bitsandbytes laya
!pip uninstall -y -q torchao
!python -m pratyaksha.world.generate --out data/wwi_in
!python -m pytest -q
!ls models

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Colab Notebooks/pratyaksha
{"customers": 60, "suppliers": 8, "invoices": 1361, "bank_entries": 997, "remittances": 501}
...................................                                      [100%]
35 passed in 22.14s
dryrun	r5_full_0.8b  r5_lora_0.8b  r5_lora_2b


## 2. R6 on the dev month (skips anything already run in Step 5)
**Why:** the gate needs each model's per-line probability on dev, to choose thresholds there and never on test.

In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"

import glob, os, json
def done(name, split, cond):
    """True if a finished run already exists (so re-running this cell never repeats GPU or API work)."""
    return any(os.path.exists(os.path.join(os.path.dirname(p), "summary.json"))
               for p in glob.glob(f"runs/*_{name}_{split}_{cond}*/predictions.jsonl"))

jobs = [("R6_control_first_option", "--system r6_control"),
        ("R6_Qwen3.5-2B", "--system r6 --opt threshold=0 --opt model=Qwen/Qwen3.5-2B --opt batch=8"),
        ("R6_r5_lora_0.8b", "--system r6 --opt threshold=0 --opt model=Qwen/Qwen3.5-0.8B --opt adapter=models/r5_lora_0.8b --opt batch=8"),
        ("R6_r5_lora_2b", "--system r6 --opt threshold=0 --opt model=Qwen/Qwen3.5-2B --opt adapter=models/r5_lora_2b --opt batch=8"),
        ("R6_r5_full_0.8b", "--system r6 --opt threshold=0 --opt model=models/r5_full_0.8b --opt batch=8")]
for cond in ["clean", "stress"]:
    for name, args in jobs:
        if done(name, "dev", cond):
            print("already done:", name, cond); continue
        !python -m pratyaksha.harness.runner {args} --split dev --condition {cond} --cost-mode gpu

/content/drive/MyDrive/Colab Notebooks/pratyaksha
config.json: 100% 795/795 [00:00<00:00, 4.43MB/s]

model.safetensors: downloading bytes:   8% 186M/2.27G [00:01<00:11, 181MB/s, 15.0MB/s  ]
model.safetensors: downloading bytes:  10% 225M/2.27G [00:02<00:11, 185MB/s, 19.8MB/s  ]
model.safetensors: downloading bytes:  20% 459M/2.27G [00:03<00:07, 252MB/s, 37.8MB/s  ]
model.safetensors: downloading bytes:  25% 565M/2.27G [00:03<00:09, 188MB/s, 46.9MB/s  ]
model.safetensors: downloading bytes:  27% 602M/2.27G [00:04<00:10, 159MB/s, 48.2MB/s  ]
model.safetensors: downloading bytes:  30% 685M/2.27G [00:04<00:12, 126MB/s, 52.6MB/s  ]
model.safetensors: downloading bytes:  38% 865M/2.27G [00:05<00:07, 194MB/s, 64.2MB/s  ]
model.safetensors: downloading bytes:  57% 1.29G/2.27G [00:07<00:03, 294MB/s, 89.8MB/s  ]
model.safetensors: downloading bytes:  61% 1.39G/2.27G [00:08<00:08, 109MB/s, 89.4MB/s  ]
model.safetensors: downloading bytes:  64% 1.46G/2.27G [00:19<00:48, 16.8MB/s, 41.9MB/s  ]
model

## 3. Laya — build training data and fine-tune (~30–60 min)
**What:** for each past bank line (months 1–4, clean + messy): the evidence (bank line, candidate parties, e-mails), the
computed options, the right option, and whether a person had to check it. Then fine-tune Laya's
*typed-decisions* checkpoint (1,024-token input) with **option shuffling**.
**Why shuffling:** the right answer is usually option M1; without shuffling Laya could learn "always pick the first",
which is just our no-AI control again.
**Why its own page matters here:** Laya is near chance before fine-tuning and ships over-confident; the trainer fits
calibration temperatures at the end, which we then check in cell 5.

In [3]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
import time, json
!python -m pratyaksha.finetune.build_laya --out data/laya/train.jsonl
!laya-train --data data/laya/train.jsonl --base typed-decisions --out models/laya_t1 --max-len 1024 --shuffle-options --device cuda --micro-batch 4 --grad-accum 16 --dry-run
t0 = time.time()
!laya-train --data data/laya/train.jsonl --base typed-decisions --out models/laya_t1 --max-len 1024 --shuffle-options --device cuda --micro-batch 4 --grad-accum 16
json.dump({"train_seconds": round(time.time() - t0, 1)}, open("models/laya_t1/train_time.json", "w"))
print("Laya training minutes:", round((time.time() - t0) / 60, 1))

/content/drive/MyDrive/Colab Notebooks/pratyaksha
Loading weights: 100% 393/393 [00:00<00:00, 6075.59it/s]
wrote 1270 rows to data/laya/train.jsonl
{
 "duplicate": 22,
 "not_offered": 102,
 "offered": 1124,
 "options=1": 106,
 "options=2": 859,
 "options=3": 258,
 "options=4": 39,
 "options=5": 8,
 "person=False": 1026,
 "person=True": 244,
 "unknown": 22
}
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/847 [00:00<?, ?B/s]           
Reconstructing (incomplete total...):  72% 847/1.18k [00:00<00:00, 3.22kB/s]
Reconstructing (incomplete total...):   0% 847/3.58M [00:00<18:31, 3.22kB/s]

Fetching 5 files:  20% 1/5 [00:00<00:01,  2.66it/s]
Reconstructing (incomplete total...):   0% 1.18k/3.59M [00:00<18:32, 3.22kB/s]
Reconstructin

## 4. Laya on the dev month (threshold 0, so every probability is stored)

In [4]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"

import glob, os, json
def done(name, split, cond):
    """True if a finished run already exists (so re-running this cell never repeats GPU or API work)."""
    return any(os.path.exists(os.path.join(os.path.dirname(p), "summary.json"))
               for p in glob.glob(f"runs/*_{name}_{split}_{cond}*/predictions.jsonl"))

for cond in ["clean", "stress"]:
    if not done("R6_laya_laya_t1", "dev", cond):
        !python -m pratyaksha.harness.runner --system r6_laya --split dev --condition {cond} --cost-mode gpu --opt threshold=0 --opt laya=models/laya_t1

/content/drive/MyDrive/Colab Notebooks/pratyaksha
Loading weights: 100% 393/393 [00:00<00:00, 4518.75it/s]
  [32/177] running accuracy 0.969
  [56/177] running accuracy 0.964
  [80/177] running accuracy 0.963
  [104/177] running accuracy 0.962
  [128/177] running accuracy 0.961
  [152/177] running accuracy 0.954
  [176/177] running accuracy 0.943
  [177/177] running accuracy 0.944
# Scorecard: R6_laya_laya_t1 · dev · clean

Run `20261123-100423_R6_laya_laya_t1_dev_clean` · n = 177

| Metric | Value |
|---|---|
| Accuracy (strict) | 94.4% |
| Handled automatically | 77.4% |
| Error rate on automatic | 0.00% |
| Latency p50 / p95 | 64.6 ms / 92.0 ms |
| ₹ per 1,000 tasks (operating) | 0.5789 |
| ₹ per 1,000 tasks (hardware amortisation) | 0.0000 |
| Tokens in / out per task | 0 / 0 |
| Data leaving the machine | 0 bytes |
| Restricted-field leaks | 0 |
| Right party among retrieved | 95.9% |
| Model output unreadable (parse errors) | 0 |
| Right answer among computed options | 94.4% |

#

## 5. The threshold knob for every model (no GPU)
**Why:** picks each model's threshold on dev (lowest that passes the business bar on clean and messy), and shows
calibration: when a model says 97%, is it right 97% of the time?

In [5]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
for m in ["R6_Qwen3.5-2B", "R6_r5_lora_0.8b", "R6_r5_lora_2b", "R6_r5_full_0.8b", "R6_laya_laya_t1"]:
    !python scripts/threshold_sweep.py --model {m}

/content/drive/MyDrive/Colab Notebooks/pratyaksha

== R6_Qwen3.5-2B · dev · clean · n=177  (accuracy 89.8%)
 threshold   alone  err alone  misbooked Rs  bank->person   bar
       0.0   76.8%      0.74%           236          100%  PASS
       0.5   71.2%      0.79%           236          100%  PASS
       0.7   66.7%      0.85%           236          100%  PASS
       0.8   53.1%      0.00%             0          100%  PASS
       0.9   27.1%      0.00%             0          100%  FAIL
      0.95   11.9%      0.00%             0          100%  FAIL
      0.97    3.4%      0.00%             0          100%  FAIL
      0.98    0.6%      0.00%             0          100%  FAIL
      0.99    0.0%      0.00%             0          100%  FAIL
     0.995    0.0%      0.00%             0          100%  FAIL
     0.999    0.0%      0.00%             0          100%  FAIL
    0.9999    0.0%      0.00%             0          100%  FAIL
  model's own needs_review flag (+ guardrail): alone 26.0%, 

## 6. R7 — Claude Opus 5.5 on the dev month
**First a 3-line check** that prints the real cost per line, then the full run (177 clean + 177 messy).
Price: $4 per million input tokens, $20 per million output tokens; Opus always thinks, and thinking is billed as output.
Answers are cached in `runs/opus_cache.jsonl`, so a re-run never pays twice. The egress meter counts every byte sent.

In [6]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
import os, json, glob
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
!python -m pratyaksha.harness.runner --system r7 --split dev --condition clean --limit 3 --cost-mode api --api-usd-in 4 --api-usd-out 20 --tag check
s = json.load(open(sorted(glob.glob("runs/*R7_claude-opus-5-5_dev_clean_check/summary.json"))[-1]))
per_line_usd = (s["tokens_in_per_task"] * 4 + s["tokens_out_per_task"] * 20) / 1e6
print(f"tokens in/out per line: {s['tokens_in_per_task']:.0f} / {s['tokens_out_per_task']:.0f}")
print(f"cost per line ≈ ${per_line_usd:.4f};  all 708 dev+test lines ≈ ${per_line_usd*708:.2f}")

/content/drive/MyDrive/Colab Notebooks/pratyaksha
Loading weights: 100% 393/393 [00:00<00:00, 3978.23it/s]
  [3/3] running accuracy 1.000
# Scorecard: R7_claude-opus-5-5 · dev · clean

Run `20261123-100719_R7_claude-opus-5-5_dev_clean_check` · n = 3

| Metric | Value |
|---|---|
| Accuracy (strict) | 100.0% |
| Handled automatically | 100.0% |
| Error rate on automatic | 0.00% |
| Latency p50 / p95 | 2217.8 ms / 2872.8 ms |
| ₹ per 1,000 tasks (operating) | 437.0667 |
| ₹ per 1,000 tasks (hardware amortisation) | 0.0000 |
| Tokens in / out per task | 762 / 96 |
| Data leaving the machine | 5,175 bytes |
| Restricted-field leaks | 0 |
| Right party among retrieved | 100.0% |
| Model output unreadable (parse errors) | 0 |
| Right answer among computed options | 100.0% |

## Business acceptance bar

| Rule | Result |
|---|---|
| Automatic entries ≤ 1% wrong (0.00%) | PASS |
| Misbooked rupees ≤ ₹1,000 (₹0.00) | PASS |
| Every unexplained entry surfaced (100%, 0 silently matched) | PASS |


**Check the estimate above.** If it fits your credit, run the next cell (about 30–60 min; it needs no GPU work except the reranker).

In [7]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"

import glob, os, json
def done(name, split, cond):
    """True if a finished run already exists (so re-running this cell never repeats GPU or API work)."""
    return any(os.path.exists(os.path.join(os.path.dirname(p), "summary.json"))
               for p in glob.glob(f"runs/*_{name}_{split}_{cond}*/predictions.jsonl"))

import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
for cond in ["clean", "stress"]:
    if not done("R7_claude-opus-5-5", "dev", cond + "_full"):
        !python -m pratyaksha.harness.runner --system r7 --split dev --condition {cond} --cost-mode api --api-usd-in 4 --api-usd-out 20 --tag full

/content/drive/MyDrive/Colab Notebooks/pratyaksha
Loading weights: 100% 393/393 [00:00<00:00, 4634.73it/s]
  [32/177] running accuracy 0.969
  [56/177] running accuracy 0.964
  [80/177] running accuracy 0.963
  [104/177] running accuracy 0.962
  [128/177] running accuracy 0.961
  [152/177] running accuracy 0.954
  [176/177] running accuracy 0.943
  [177/177] running accuracy 0.944
# Scorecard: R7_claude-opus-5-5 · dev · clean

Run `20261123-101132_R7_claude-opus-5-5_dev_clean_full` · n = 177

| Metric | Value |
|---|---|
| Accuracy (strict) | 94.4% |
| Handled automatically | 70.6% |
| Error rate on automatic | 0.00% |
| Latency p50 / p95 | 2408.0 ms / 4940.2 ms |
| ₹ per 1,000 tasks (operating) | 519.8344 |
| ₹ per 1,000 tasks (hardware amortisation) | 0.0000 |
| Tokens in / out per task | 717 / 152 |
| Data leaving the machine | 289,450 bytes |
| Restricted-field leaks | 0 |
| Right party among retrieved | 95.9% |
| Model output unreadable (parse errors) | 0 |
| Right answer among co

## 7. THE FINAL TEST MONTH — every system once (~60–90 min)
**Why:** the acceptance bar was frozen on 23 Nov and every threshold was chosen on dev. Month 6 has never been used,
so these are the honest numbers for the report. Nothing is changed after this cell runs.

In [8]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"

import glob, os, json
def done(name, split, cond):
    """True if a finished run already exists (so re-running this cell never repeats GPU or API work)."""
    return any(os.path.exists(os.path.join(os.path.dirname(p), "summary.json"))
               for p in glob.glob(f"runs/*_{name}_{split}_{cond}*/predictions.jsonl"))

import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
def thr(m):
    f = f"runs/threshold_{m}.json"
    t = json.load(open(f))["threshold"] if os.path.exists(f) else None
    return 0.999 if t is None else t
T = {m: thr(m) for m in ["R6_Qwen3.5-2B", "R6_r5_lora_0.8b", "R6_r5_lora_2b", "R6_r5_full_0.8b", "R6_laya_laya_t1"]}
print("thresholds chosen on dev:", T)
json.dump(T, open("runs/thresholds_used_on_test.json", "w"), indent=1)
jobs = [("rules", "--system rules"),
        ("R3_control_first_option", "--system r3_control"),
        ("R6_control_first_option", "--system r6_control"),
        ("R5_r5_lora_0.8b", "--system r5 --opt model=Qwen/Qwen3.5-0.8B --opt adapter=models/r5_lora_0.8b --opt batch=8"),
        ("R6_Qwen3.5-2B", f"--system r6 --opt threshold={T['R6_Qwen3.5-2B']} --opt model=Qwen/Qwen3.5-2B --opt batch=8"),
        ("R6_r5_lora_0.8b", f"--system r6 --opt threshold={T['R6_r5_lora_0.8b']} --opt model=Qwen/Qwen3.5-0.8B --opt adapter=models/r5_lora_0.8b --opt batch=8"),
        ("R6_r5_lora_2b", f"--system r6 --opt threshold={T['R6_r5_lora_2b']} --opt model=Qwen/Qwen3.5-2B --opt adapter=models/r5_lora_2b --opt batch=8"),
        ("R6_r5_full_0.8b", f"--system r6 --opt threshold={T['R6_r5_full_0.8b']} --opt model=models/r5_full_0.8b --opt batch=8"),
        ("R6_laya_laya_t1", f"--system r6_laya --opt threshold={T['R6_laya_laya_t1']} --opt laya=models/laya_t1"),
        ("R7_claude-opus-5-5", "--system r7 --cost-mode api --api-usd-in 4 --api-usd-out 20")]
for cond in ["clean", "stress"]:
    for name, args in jobs:
        if done(name, "test", cond):
            print("already done:", name, cond); continue
        mode = "" if "--cost-mode" in args else "--cost-mode gpu"
        !python -m pratyaksha.harness.runner {args} {mode} --split test --condition {cond}

/content/drive/MyDrive/Colab Notebooks/pratyaksha
thresholds chosen on dev: {'R6_Qwen3.5-2B': 0.0, 'R6_r5_lora_0.8b': 0.0, 'R6_r5_lora_2b': 0.0, 'R6_r5_full_0.8b': 0.0, 'R6_laya_laya_t1': 0.0}
  [25/185] running accuracy 0.760
  [50/185] running accuracy 0.780
  [75/185] running accuracy 0.800
  [100/185] running accuracy 0.750
  [125/185] running accuracy 0.752
  [150/185] running accuracy 0.753
  [175/185] running accuracy 0.754
  [185/185] running accuracy 0.746
# Scorecard: rules · test · clean

Run `20261123-103225_rules_test_clean` · n = 185

| Metric | Value |
|---|---|
| Accuracy (strict) | 74.6% |
| Handled automatically | 67.6% |
| Error rate on automatic | 0.00% |
| Latency p50 / p95 | 135.1 ms / 268.0 ms |
| ₹ per 1,000 tasks (operating) | 1.1197 |
| ₹ per 1,000 tasks (hardware amortisation) | 0.0000 |
| Tokens in / out per task | 0 / 0 |
| Data leaving the machine | 0 bytes |
| Restricted-field leaks | 0 |
| Right party among retrieved | n/a |
| Model output unreadable (pa

## 8. The results table
**Why:** the headline result of the project — test month, every system.

In [9]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!python scripts/compare.py --split test
for m in ["R6_r5_lora_0.8b", "R6_r5_lora_2b", "R6_laya_laya_t1", "R7_claude-opus-5-5"]:
    !python scripts/error_analysis.py --model {m} --split test

/content/drive/MyDrive/Colab Notebooks/pratyaksha
system                      cond        n     acc    auto  autoErr   p50 s  party@k  tok_in parseErr   bar
rules                       clean     185   74.6%   67.6%     0.0%    0.14        -       0        0  FAIL
R3_control_first_option     clean     185   89.7%   81.6%     0.0%    0.10    93.8%     342        0  FAIL
R6_control_first_option     clean     185   89.7%   74.1%     0.0%    0.10    93.8%     342        0  PASS
R5_r5_lora_0.8b             clean     185   91.4%   80.0%     0.0%    0.32    93.8%     530        0  PASS
R6_Qwen3.5-2B               clean     185   87.0%   73.5%     0.7%    0.68    93.8%     530        1  PASS
R6_r5_lora_0.8b             clean     185   91.4%   75.7%     0.0%    0.34    93.8%     530        0  PASS
R6_r5_lora_2b               clean     185   91.4%   75.7%     0.0%    0.41    93.8%     530        0  PASS
R6_r5_full_0.8b             clean     185   91.4%   75.7%     0.0%    0.34    93.8%     530   

In [10]:
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!python scripts/significance.py --split test
!python scripts/significance.py --split test --ref R7_claude-opus-5-5

/content/drive/MyDrive/Colab Notebooks/pratyaksha

== test · clean · reference R6_control_first_option
system                          n     acc  ref acc  fixes  breaks       p   95% CI of difference
R3_control_first_option       185   89.7%    89.7%      0       0   1.000   [+0.0%, +0.0%]
R5_r5_lora_0.8b               185   91.4%    89.7%      3       0   0.250   [+0.0%, +3.8%]
R6_Qwen3.5-2B                 185   87.0%    89.7%      3       8   0.227   [-5.9%, +0.5%]
R6_laya_laya_t1               185   91.4%    89.7%      3       0   0.250   [+0.0%, +3.8%]
R6_r5_full_0.8b               185   91.4%    89.7%      3       0   0.250   [+0.0%, +3.8%]
R6_r5_lora_0.8b               185   91.4%    89.7%      3       0   0.250   [+0.0%, +3.8%]
R6_r5_lora_2b                 185   91.4%    89.7%      3       0   0.250   [+0.0%, +3.8%]
R7_claude-opus-5-5            185   89.7%    89.7%      3       3   1.000   [-2.7%, +2.7%]
rules                         185   74.6%    89.7%      0      28   0.00

## 9. Send back
*File → Download → .ipynb*, send it, then *Runtime → Disconnect and delete runtime*.